MLR-503: On-Edge Mobile Diagnosis of Malaria Parasites
using Machine Learning

<br>

Authored by: Meriem Aoudia, Raghad Aldamani, Diaa Abuhani


## **Data Preprocessing**

In [ ]:
import pandas as pd
import os

folder_path = "drive/MyDrive/Colab Notebooks/MLR503_Project/Classes/"


dic={'filename':[],'label':[]}

for lbl,folder in enumerate(os.listdir(folder_path)):
    for file in os.listdir(folder_path+ "/" + folder):
        dic['filename'].append(folder+'/'+file)
        dic['label'].append(lbl)

allfilenames=pd.DataFrame(dic)
allfilenames['label']=allfilenames['label'].astype(str)
allfilenames

,filename,label
0,Positive/id_j0zahs4ik5.jpg,0
1,Positive/id_dmyxhehyfz.jpg,0
2,Positive/id_f1e3aetgv3.jpg,0
3,Positive/id_wft4oplzxm.jpg,0
4,Positive/id_dywyswudfa.jpg,0
...,...,...
3201,Negative/id_hx2ab922zc.jpg,2
3202,Negative/id_zzmhan693n.jpg,2
3203,Negative/id_18vohhd1kx.jpg,2
3204,Negative/id_awqa79vqcu.jpg,2


In [ ]:
list(enumerate(os.listdir(folder_path)))

[(0, 'Positive'), (1, 'Suspicious'), (2, 'Negative')]

In [ ]:
import tensorflow as tf

img_height,img_width=384,384
batch_size=32

def preprocess_data(image, label):

    image = tf.image.convert_image_dtype(image, tf.float32)
    label = tf.one_hot(label, 3)

    return image, label

train_ds = tf.keras.preprocessing.image_dataset_from_directory(
  folder_path + "/",
  validation_split=0.2,
  subset="training",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

train_ds = train_ds.map(preprocess_data)

Found 3206 files belonging to 3 classes.
Using 2565 files for training.


In [ ]:
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
  folder_path + "/",
  validation_split=0.2,
  subset="validation",
  seed=123,
  image_size=(img_height, img_width),
  batch_size=batch_size)

val_ds = test_ds.map(preprocess_data)

Found 3206 files belonging to 3 classes.
Using 641 files for validation.


### **Model fitting**

In [ ]:
from tensorflow.keras.applications import NASNetMobile
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten

pretrained_model = NASNetMobile(
    include_top=False,
    input_shape=(384, 384, 3),
    pooling='avg',
    weights='imagenet'
)

for layer in pretrained_model.layers:
    layer.trainable = False

nasnet_model = Sequential([
    pretrained_model,
    Flatten(),
    Dense(512, activation='relu'),
    Dense(3, activation='softmax')
])

nasnet_model.summary()

19993432/19993432 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ nasnet_mobile (Functional)      │ (None, 1056)           │     4,269,716 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1056)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       541,184 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,812,439 (18.36 MB)

 Trainable params: 542,723 (2.07 MB)

 Non-trainable params: 4,269,716 (16.29 MB)

In [ ]:
nasnet_model.compile(optimizer=Adam(learning_rate=0.001),loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
epochs=15
history = nasnet_model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

Epoch 1/15


I0000 00:00:1733164519.978728 2727509 service.cc:152] XLA service 0x72b088002c90 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1733164519.978763 2727509 service.cc:160]   StreamExecutor device (0): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
I0000 00:00:1733164519.978771 2727509 service.cc:160]   StreamExecutor device (1): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
I0000 00:00:1733164519.978774 2727509 service.cc:160]   StreamExecutor device (2): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
I0000 00:00:1733164519.978777 2727509 service.cc:160]   StreamExecutor device (3): NVIDIA A100-SXM4-80GB, Compute Capability 8.0
2024-12-02 22:35:23.092474: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1733164534.133667 2727509 cuda_dnn.cc:529] Loaded cuDNN version 90501
2024-12-02 22:35:35.191357: I external/local_xla

 3/81 ━━━━━━━━━━━━━━━━━━━━ 2s 37ms/step - accuracy: 0.4514 - loss: 8.8413   

I0000 00:00:1733164546.090940 2727509 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


79/81 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.7043 - loss: 2.1139

2024-12-02 22:35:53.420536: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:344] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22850', 4 bytes spill stores, 4 bytes spill loads

2024-12-02 22:35:53.638616: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:344] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22843', 4 bytes spill stores, 4 bytes spill loads

2024-12-02 22:35:53.658819: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:344] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_22843', 304 bytes spill stores, 304 bytes spill loads

2024-12-02 22:35:54.235888: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-02 22:35:54.401602: E external/local

81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 179ms/step - accuracy: 0.7065 - loss: 2.0847

2024-12-02 22:36:14.120986: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-02 22:36:14.283128: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-02 22:36:14.442751: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-02 22:36:14.753351: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-02 22:36:14.909192: E external/local_xla/xla/stream_

81/81 ━━━━━━━━━━━━━━━━━━━━ 74s 424ms/step - accuracy: 0.7076 - loss: 2.0707 - val_accuracy: 0.9080 - val_loss: 0.2480
Epoch 2/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 96ms/step - accuracy: 0.8908 - loss: 0.2907 - val_accuracy: 0.8378 - val_loss: 0.5352
Epoch 3/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 92ms/step - accuracy: 0.8919 - loss: 0.3564 - val_accuracy: 0.8861 - val_loss: 0.3074
Epoch 4/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 96ms/step - accuracy: 0.9141 - loss: 0.2472 - val_accuracy: 0.8331 - val_loss: 0.4335
Epoch 5/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 95ms/step - accuracy: 0.8949 - loss: 0.2597 - val_accuracy: 0.9267 - val_loss: 0.1789
Epoch 6/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 7s 80ms/step - accuracy: 0.9301 - loss: 0.1692 - val_accuracy: 0.8955 - val_loss: 0.2623
Epoch 7/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 7s 87ms/step - accuracy: 0.9397 - loss: 0.1550 - val_accuracy: 0.9142 - val_loss: 0.2591
Epoch 8/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 7s 91ms/step - accuracy: 0.9444 - loss: 0.1437 - val_accuracy: 0.8253 - val_loss: 

In [ ]:
nasnet_model.save('nasnet_model.keras')

### **Qunatization**

In [ ]:
#@title QAT
import tensorflow_model_optimization as tfmot
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam


def apply_quantization(layer):
        if isinstance(layer, tf.keras.layers.Conv2D):
            return tfmot.quantization.keras.quantize_annotate_layer(layer)
        return layer

model_pre = tf.keras.applications.NASNetMobile(
    include_top=False,
    input_shape=(384, 384, 3),
    pooling='avg',
    weights='imagenet'
)

for layer in model_pre.layers:
    layer.trainable = False

inputs = model_pre.input
x = Flatten()(model_pre.output)
x = Dense(512, activation='relu')(x)
outputs = Dense(3, activation='softmax')(x)
concatenated_model = tf.keras.Model(inputs=inputs, outputs=outputs)

annotated_model = tf.keras.models.clone_model(
    concatenated_model,
    clone_function=apply_quantization,)

annotated_model.compile(optimizer=Adam(learning_rate=0.001),loss='categorical_crossentropy',metrics=['accuracy'])

You can see that the layers are QuantizeAnnotated

In [ ]:
annotated_model.fit()

Model: "model_3"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_2 (InputLayer)        [(None, 384, 384, 3)]        0         []                            
                                                                                                  
 quantize_annotate_105 (Qua  (None, 192, 192, 32)         864       ['input_2[0][0]']             
 ntizeAnnotate)                                                                                   
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 192, 192, 32)         128       ['quantize_annotate_105[0][0]'
 on)                                                                ]                             
                                                                                            

In [ ]:
epochs=15
history = annotated_model.fit(
  train_ds,
  validation_data=val_ds,
  epochs=epochs
)

Epoch 1/15


2024-12-03 01:50:44.775317: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:344] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1393', 16 bytes spill stores, 16 bytes spill loads

2024-12-03 01:50:44.795199: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:344] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4291', 32 bytes spill stores, 40 bytes spill loads

2024-12-03 01:50:44.846507: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:344] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4291', 24 bytes spill stores, 24 bytes spill loads

2024-12-03 01:50:44.857794: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:344] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_4291', 16 bytes spill stores, 20 bytes spill loads

2024-12-03 01:50:44.897507: I external/l

77/81 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.8294 - loss: 0.4586

2024-12-03 01:50:59.460827: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:50:59.622735: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:51:00.110922: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:51:00.311391: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:51:00.759022: E external/local_xla/xla/stream_

81/81 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step - accuracy: 0.8331 - loss: 0.4482

2024-12-03 01:51:11.700484: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:51:11.856947: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:51:12.648095: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:51:12.798613: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2024-12-03 01:51:13.104903: E external/local_xla/xla/stream_

81/81 ━━━━━━━━━━━━━━━━━━━━ 41s 247ms/step - accuracy: 0.8339 - loss: 0.4458 - val_accuracy: 0.9579 - val_loss: 0.1093
Epoch 2/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 94ms/step - accuracy: 0.9686 - loss: 0.1009 - val_accuracy: 0.9688 - val_loss: 0.0808
Epoch 3/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 93ms/step - accuracy: 0.9734 - loss: 0.0720 - val_accuracy: 0.9657 - val_loss: 0.0796
Epoch 4/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 7s 84ms/step - accuracy: 0.9759 - loss: 0.0601 - val_accuracy: 0.9548 - val_loss: 0.1097
Epoch 5/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 92ms/step - accuracy: 0.9771 - loss: 0.0632 - val_accuracy: 0.9594 - val_loss: 0.1033
Epoch 6/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 7s 81ms/step - accuracy: 0.9735 - loss: 0.0636 - val_accuracy: 0.9735 - val_loss: 0.0532
Epoch 7/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 7s 81ms/step - accuracy: 0.9919 - loss: 0.0280 - val_accuracy: 0.9797 - val_loss: 0.0482
Epoch 8/15
81/81 ━━━━━━━━━━━━━━━━━━━━ 8s 93ms/step - accuracy: 0.9856 - loss: 0.0334 - val_accuracy: 0.9688 - val_loss: 

In [ ]:
concatenated_model.save('drive/MyDrive/Colab Notebooks/MLR503_Project/Results/QAT/Nasnet_model.h5')

In [ ]:
import pathlib
tflite_model_file = pathlib.Path("drive/MyDrive/Colab Notebooks/MLR503_Project/Results/QAT/resnet50_QAT_f16.tflite")

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_model_file))
interpreter.allocate_tensors()

In [ ]:
import time

y_pred = []
inference_times = []

for image, label in zip(x_imgs, y_true):
    start_time = time.time()

    interpreter = tf.lite.Interpreter(model_path=str(tflite_model_file))
    interpreter.allocate_tensors()

    test_image = np.expand_dims(image, axis=0).astype(np.float32)

    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    interpreter.set_tensor(input_index, test_image)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_index)


    end_time = time.time()
    inference_time = (end_time - start_time) * 1000
    inference_times.append(inference_time)
    y_pred.extend(np.argmax(predictions, axis=1))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

average_inference_time_ms = sum(inference_times) / len(y_true)
average_inference_time_ms

print(f"Avg Inference Time: {average_inference_time_ms}")

Avg Inference Time: 851.7620239912441


In [ ]:
# Assuming you have true labels stored in y_true and predicted labels stored in y_pred

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred)

# Calculate precision
precision = precision_score(y_true, y_pred, average='weighted')

# Calculate recall
recall = recall_score(y_true, y_pred, average='weighted')

# Calculate F1-score
f1 = f1_score(y_true, y_pred, average='weighted')

# Print the results
print(f"Accuracy: {accuracy:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")
print(f"F1-score: {f1:.5f}")

Accuracy: 0.99064
Precision: 0.99060
Recall: 0.99064
F1-score: 0.99060


## **FP16**

In [ ]:
model =  tf.keras.models.load_model("drive/MyDrive/Colab Notebooks/MLR503_Project/Results/F32/nasnet_model.h5")
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]
Tflite_quanit_model = converter.convert()

Saved artifact at '/tmp/tmp9p6moycs'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 384, 384, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136707985193648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707834403776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707834402016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707985190480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707985191536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707985188192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707985229952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707985228544: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707985228368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707985229776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1367079852345

In [ ]:
import pathlib
tflite_model_file = pathlib.Path("drive/MyDrive/Colab Notebooks/MLR503_Project/Results/F16/nasnet_model.tflite")
tflite_model_file.write_bytes(Tflite_quanit_model)

9799564

In [ ]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_model_file))
interpreter.allocate_tensors()

In [ ]:
import time

y_pred = []
inference_times = []

for image, label in zip(x_imgs, y_true):
    start_time = time.time()

    interpreter = tf.lite.Interpreter(model_path=str(tflite_model_file))
    interpreter.allocate_tensors()

    test_image = np.expand_dims(image, axis=0).astype(np.float32)

    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    interpreter.set_tensor(input_index, test_image)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_index)


    end_time = time.time()
    inference_time = (end_time - start_time) * 1000
    inference_times.append(inference_time)
    y_pred.extend(np.argmax(predictions, axis=1))

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

average_inference_time_ms = sum(inference_times) / len(y_true)
average_inference_time_ms

print(f"Avg Inference Time: {average_inference_time_ms/1000:.5f}")

Avg Inference Time: 0.25523


In [ ]:
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

print(f"Accuracy: {accuracy:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")
print(f"F1-score: {f1:.5f}")

Accuracy: 0.96412
Precision: 0.96695
Recall: 0.96412
F1-score: 0.96483


### **Represntatitive Learning**

In [ ]:
def representative_data_gen():
    for input_value, _ in train_ds.unbatch().batch(1).take(100):
        yield [input_value]


model_path = "drive/MyDrive/Colab Notebooks/MLR503_Project/Results/F32/nasnet_model.h5"
model = tf.keras.models.load_model(model_path)

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen

# Ensure that if any ops can't be quantized, the converter throws an error
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]

# Setting the input and output types to uint8
converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

# Convert the model
tflite_model_quant = converter.convert()

Saved artifact at '/tmp/tmp1zt4t7dw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 384, 384, 3), dtype=tf.float32, name='input_layer_1')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  136707824322336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677106272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677106976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677102048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677104336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677109440: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677103632: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677107328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677105568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136707677100464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1367076771052

/usr/local/lib/python3.10/dist-packages/tensorflow/lite/python/convert.py:983: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


In [ ]:
# Save the converted model
tflite_model_file = pathlib.Path("drive/MyDrive/Colab Notebooks/MLR503_Project/Results/INT/nasnet_model.tflite")
tflite_model_file.write_bytes(tflite_model_quant)

# Load the TFLite model for inference
interpreter = tf.lite.Interpreter(model_path=str(tflite_model_file))
interpreter.allocate_tensors()

In [ ]:
import time

y_pred = []
inference_times = []

for image, label in zip(x_imgs, y_true):
    start_time = time.time()

    interpreter = tf.lite.Interpreter(model_path=str(tflite_model_file))
    interpreter.allocate_tensors()

    test_image = np.expand_dims(image, axis=0).astype(np.uint8)

    input_index = interpreter.get_input_details()[0]["index"]
    output_index = interpreter.get_output_details()[0]["index"]

    interpreter.set_tensor(input_index, test_image)
    interpreter.invoke()
    predictions = interpreter.get_tensor(output_index)


    end_time = time.time()
    inference_time = (end_time - start_time) * 1000
    inference_times.append(inference_time)
    y_pred.extend(np.argmax(predictions, axis=1))

In [ ]:
# Calculate average inference time
average_inference_time_ms = sum(inference_times) / len(y_true)
print(f"Avg Inference Time: {average_inference_time_ms}")

Avg Inference Time: 692.435498914555


In [ ]:
# Calculate evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='weighted')
f1 = f1_score(y_true, y_pred, average='weighted')

# Print the results
print(f"Accuracy: {accuracy:.5f}")
print(f"Precision: {precision:.5f}")
print(f"Recall: {recall:.5f}")
print(f"F1-score: {f1:.5f}")

Accuracy: 0.60218
Precision: 0.71080
Recall: 0.60218
F1-score: 0.64059
